In [0]:
# 04_gold – fachliche Kennzahlen für Reporting
from pyspark.sql import functions as F, Window

DB = "workspace.banking"
tx = spark.table(f"{DB}.silver_transactions")
cust = spark.table(f"{DB}.silver_customers")
acc = spark.table(f"{DB}.silver_accounts")
txc = tx.join(cust.select("customer_id", "segment"), "customer_id")

# 1) Monats-KPIs je Kundensegment
monthly = (txc
    .withColumn("month", F.date_trunc("month", "booking_date").cast("date"))
    .groupBy("month", "segment")
    .agg(F.count("*").alias("tx_count"),
         F.sum(F.when(F.col("signed_amount_eur") > 0, F.col("signed_amount_eur")).otherwise(0)).alias("inflow_eur"),
         F.sum(F.when(F.col("signed_amount_eur") < 0, -F.col("signed_amount_eur")).otherwise(0)).alias("outflow_eur"),
         F.sum("signed_amount_eur").alias("net_flow_eur"),
         F.round(F.avg("amount_eur"), 2).alias("avg_ticket_eur")))
monthly.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.gold_monthly_kpis")

# 2) Kundensicht (Customer 360)
c360 = (cust
    .join(acc.groupBy("customer_id").agg(F.count("*").alias("num_accounts")), "customer_id", "left")
    .join(tx.groupBy("customer_id").agg(
             F.count("*").alias("tx_count"),
             F.sum("amount_eur").alias("total_volume_eur"),
             F.round(F.avg("amount_eur"), 2).alias("avg_amount_eur"),
             F.max("booking_date").alias("last_tx_date")), "customer_id", "left"))
c360.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.gold_customer_360")

# 3) Auffällige Transaktionen: Betrag > 3x Kundendurchschnitt UND > 1.000 EUR
w = Window.partitionBy("customer_id")
suspicious = (txc
    .withColumn("customer_avg_eur", F.round(F.avg("amount_eur").over(w), 2))
    .withColumn("factor", F.round(F.col("amount_eur") / F.col("customer_avg_eur"), 1))
    .filter((F.col("amount_eur") > 3 * F.col("customer_avg_eur")) & (F.col("amount_eur") > 1000))
    .select("transaction_id", "customer_id", "segment", "booking_date", "amount_eur",
            "customer_avg_eur", "factor", "transaction_type", "channel"))
suspicious.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.gold_suspicious_transactions")

# 4) Datenqualitäts-Report
dq = (spark.table(f"{DB}.quarantine_transactions")
    .withColumn("rule", F.explode(F.split("dq_errors", ",")))
    .groupBy("rule").agg(F.count("*").alias("error_count")))
dq.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.gold_data_quality")

print("Gold fertig. Auffällige Transaktionen:", suspicious.count())